In [ ]:
# BUSINESS SCIENCE -----
# LAB 91: CUSTOMER LIFETIME VALUE -----
# Part 3: PREDICTIVE CLV MODELS (ADVANCED) -----
# *** -----

# LIBRARIES -----
import pandas as pd
import pytimetk as tk
import pycaret.classification as clf
import pycaret.regression as reg

# CONSTANTS -----
profit_margin = 0.15 # 15% Profit on Products

# DATA -----
transactions_df = pd.read_csv('data/transactions_processed.csv')
df = transactions_df.copy()
df['timestamp'] = pd.to_datetime(df['timestamp'])

# MACHINE LEARNING -----
# Frame the problem:
# - What will the customers spend in the next 90-Days? (Regression)
# - What is the probability of a customer to make a purchase in next 90-days? (Classification)

n_days = 90
max_date = df['timestamp'].max()
cutoff = max_date - pd.to_timedelta(n_days, unit="d")

# Train-Test Split
temporal_in_df  = df[df['timestamp'] < cutoff]
temporal_out_df = df[df['timestamp'] >= cutoff] \
    .query('household_key in @temporal_in_df.household_key')

# FEATURE ENGINEERING -----

# Make Targets from out data ----
targets_df = temporal_out_df[['household_key', 'timestamp', 'sales_value']] \
    .groupby('household_key') \
    .sum() \
    .rename({'sales_value': 'sales_90_value'}, axis=1) \
    .assign(sales_90_flag = 1)

# Make Recency (Date) Features from in data ----
max_date = temporal_in_df['timestamp'].max()

recency_features_df = temporal_in_df \
    [['household_key', 'timestamp']] \
    .groupby('household_key') \
    .apply(lambda x: int((x['timestamp'].max() - max_date) / pd.to_timedelta(1, "day"))) \
    .to_frame() \
    .set_axis(["recency"], axis=1)

# Make Frequency (Count) Features from in data ----
frequency_features_df = temporal_in_df \
    [['household_key', 'timestamp']] \
    .groupby('household_key') \
    .count() \
    .set_axis(['frequency'], axis=1)

# Make Monetary Features from in data ----
monetary_features_df = temporal_in_df \
    .groupby('household_key') \
    .aggregate({'sales_value': ["sum", "mean"]}) \
    .set_axis(['sales_value_sum', 'sales_value_mean'], axis=1)

# OTHER FEATURES ----

# Transactions Last Month
cutoff_28d = cutoff - pd.to_timedelta(28, unit="d")

transactions_last_month_df = temporal_in_df[['household_key', 'timestamp']] \
    .drop_duplicates() \
    .query('timestamp >= @cutoff_28d') \
    .groupby('household_key') \
    .size() \
    .to_frame() \
    .set_axis(['transactions_last_month'], axis=1)

# Transactions Last 2 Weeks
cutoff_14d = cutoff - pd.to_timedelta(14, unit="d")

transactions_last_2weeks_df = temporal_in_df[['household_key', 'timestamp']] \
    .drop_duplicates() \
    .query('timestamp >= @cutoff_14d') \
    .groupby('household_key') \
    .size() \
    .to_frame() \
    .set_axis(['transactions_last_2weeks'], axis=1)

# Spend Last 2 Weeks
sales_last_2weeks_df = temporal_in_df[['household_key', 'sales_value', 'timestamp']] \
    .query('timestamp >= @cutoff_14d') \
    .groupby('household_key') \
    ['sales_value'].sum() \
    .to_frame() \
    .set_axis(['sales_value_last_2weeks'], axis=1)

# COMBINE FEATURES ----
features_df = pd.concat([
    recency_features_df, 
    frequency_features_df,
    monetary_features_df, 
    transactions_last_month_df,
    transactions_last_2weeks_df, 
    sales_last_2weeks_df
], axis=1) \
    .merge(targets_df, left_index=True, right_index=True, how="left") \
    .fillna(0)

# MACHINE LEARNING -----

# REGRESSION ----
reg_setup = reg.setup(
    data          = features_df.drop('sales_90_flag', axis=1),
    target        = 'sales_90_value',
    train_size    = 0.8,
    normalize     = True,
    session_id    = 123,
    verbose       = True,
    log_experiment= False
)

xgb_reg_model = reg.create_model('xgboost')

reg_predictions_df = reg.predict_model(xgb_reg_model, data=features_df) \
    .sort_values('prediction_label', ascending=False)

# CLASSIFICATION (SPEND PROBABILITY) ----
clf_setup = clf.setup(
    data          = features_df.drop('sales_90_value', axis=1),
    target        = 'sales_90_flag',
    train_size    = 0.8,
    session_id    = 123,
    verbose       = True,
    log_experiment= False
)

xgb_clf_model = clf.create_model('xgboost')

clf_predictions_df = clf.predict_model(xgb_clf_model, data=features_df, raw_score=True) \
    .sort_values('prediction_score_1', ascending=False)

# EXTRACTING INSIGHTS ----
reg.interpret_model(xgb_reg_model)
clf.interpret_model(xgb_clf_model)

# BUSINESS VALUE ----
# What would happen if you could increase revenue by 10%?
# reg_predictions_df['prediction_label'].sum() 

top_20_customers = reg_predictions_df.head(20).index.tolist()

# Increase *FREQUENCY* of purchases: sell them these
transactions_df \
    .query('household_key in @top_20_customers') \
    .groupby('commodity_desc') \
    .size() \
    .to_frame() \
    .set_axis(['count'], axis=1) \
    .sort_values('count', ascending=False)

# Increase *SIZE* of purchases: sell them these
transactions_df \
    [['household_key', 'commodity_desc', 'sales_value']] \
    .query('household_key in @top_20_customers') \
    .groupby('commodity_desc') \
    .sum() \
    .sort_values('sales_value', ascending=False)